## ETL of GeoSpatial data from US Census to Hugging Face

The US Census has restrictions on how it serves files.  So we'll copy them elsewhere first.


## Setup


In [14]:
%%capture
%%bash
apt-get update
apt-get install -y tree jq


In [2]:
import os
import pandas as pd
from google.colab import userdata


## Get a list of all shapefiles


In [3]:
url = 'https://www2.census.gov/geo/tiger/TIGER2024/ROADS/'
url


'https://www2.census.gov/geo/tiger/TIGER2024/ROADS/'

In [4]:
tables = pd.read_html( url )
len(tables)


1

In [5]:
df = tables[0]
df


,Unnamed: 0,Name,Last modified,Size,Description
0,NaN,NaN,NaN,NaN,NaN
1,NaN,Parent Directory,NaN,-,NaN
2,NaN,tl_2024_01001_roads.zip,2025-06-27 16:04,937K,NaN
3,NaN,tl_2024_01003_roads.zip,2025-06-27 16:04,2.2M,NaN
4,NaN,tl_2024_01005_roads.zip,2025-06-27 16:04,667K,NaN
...,...,...,...,...,...
3231,NaN,tl_2024_72153_roads.zip,2025-06-27 16:04,456K,NaN
3232,NaN,tl_2024_78010_roads.zip,2025-06-27 16:04,1.0M,NaN
3233,NaN,tl_2024_78020_roads.zip,2025-06-27 16:04,287K,NaN
3234,NaN,tl_2024_78030_roads.zip,2025-06-27 16:04,723K,NaN


In [6]:
shapefiles = [ x for x in df["Name"].dropna() if x.endswith("zip") ]
shapefiles[:5]


['tl_2024_01001_roads.zip',
 'tl_2024_01003_roads.zip',
 'tl_2024_01005_roads.zip',
 'tl_2024_01007_roads.zip',
 'tl_2024_01009_roads.zip']

In [7]:
with open("shapefiles", "w") as f:
  f.write( "\n".join( [ url + x for x in shapefiles ] ) )


In [8]:
! ls -l

total 240
drwxr-xr-x 1 root root   4096 Nov 18 15:58 sample_data
-rw-r--r-- 1 root root 239241 Nov 19 20:59 shapefiles


## Download shapefiles


In [9]:
! mkdir -p TIGER2024/ROADS/
! mkdir -p TIGER2024/STATE/


In [15]:
! tmux new-session -s shapefiles -d 'cd TIGER2024/ROADS/ ; time -p cat ../../shapefiles | xargs -n 1 -P 20 -t curl -s -O -L ; echo == done ; sleep 1000 '


In [11]:
states_url = "https://www2.census.gov/geo/tiger/TIGER2024/STATE/tl_2024_us_state.zip"
states_url


'https://www2.census.gov/geo/tiger/TIGER2024/STATE/tl_2024_us_state.zip'

In [12]:
! cd TIGER2024/STATE/ ; curl -L -O {states_url}


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 9721k  100 9721k    0     0  53.8M      0 --:--:-- --:--:-- --:--:-- 53.9M


In [16]:
! tree TIGER2024/


TIGER2024/
├── ROADS
│   ├── tl_2024_01001_roads.zip
│   ├── tl_2024_01003_roads.zip
│   ├── tl_2024_01005_roads.zip
│   ├── tl_2024_01007_roads.zip
│   ├── tl_2024_01009_roads.zip
│   ├── tl_2024_01011_roads.zip
│   ├── tl_2024_01013_roads.zip
│   ├── tl_2024_01015_roads.zip
│   ├── tl_2024_01017_roads.zip
│   ├── tl_2024_01019_roads.zip
│   ├── tl_2024_01021_roads.zip
│   ├── tl_2024_01023_roads.zip
│   ├── tl_2024_01025_roads.zip
│   ├── tl_2024_01027_roads.zip
│   ├── tl_2024_01029_roads.zip
│   ├── tl_2024_01031_roads.zip
│   ├── tl_2024_01033_roads.zip
│   ├── tl_2024_01035_roads.zip
│   ├── tl_2024_01037_roads.zip
│   ├── tl_2024_01039_roads.zip
│   ├── tl_2024_01041_roads.zip
│   ├── tl_2024_01043_roads.zip
│   ├── tl_2024_01045_roads.zip
│   ├── tl_2024_01047_roads.zip
│   ├── tl_2024_01049_roads.zip
│   ├── tl_2024_01051_roads.zip
│   ├── tl_2024_01053_roads.zip
│   ├── tl_2024_01055_roads.zip
│   ├── tl_2024_01057_roads.zip
│   ├── tl_2024_01059_roads.zip
│   ├── tl_2024_010

In [ ]:
! find ./TIGER2024/

## Push to Hugging Face


## Configure settings and authenticate to Hugging Face


In [17]:
hf_account = os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account


'rwcitek'

In [18]:
hf_repo = os.environ["HF_REPO"] = "DUC-2026"
hf_repo


'DUC-2026'

In [19]:
hf_file = os.environ["HF_FILE"] = "TIGER2024/"
hf_file


'TIGER2024/'

In [20]:
# Retrieve the token from Colab's secret manager
_ = hf_token = os.environ["HF_TOKEN"] = userdata.get('hf_token')
f"{_[:3]} ... {_[-4:]}"


'hf_ ... dQPb'

In [21]:
# Log in automatically without an interactive prompt
! hf auth login --token ${HF_TOKEN}


Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Colab` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Push the parquet file to Hugging Face


In [25]:
%%capture hf_upload
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

# Comment out the echo to make this live
# echo DRY-RUN: \
hf upload \
  --type dataset \
  "${TARGET_REPO}" \
  "${LOCAL_SOURCE}" \
  "${REMOTE_DESTINATION}"


In [26]:
print(hf_upload.stdout)


✓ Uploaded
  url: https://huggingface.co/datasets/rwcitek/DUC-2026/commit/ab702024fe20a3821c5c2dc68c7f9428f787d7c9



In [27]:
%%capture hf_repo_list
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

# Comment out the echo to make this live
# echo DRY-RUN: \
hf datasets  \
  list -R \
  "${TARGET_REPO}" \


In [28]:
print(hf_repo_list.stdout)


              2026-11-19 21:09:26  TIGER2024/
              2026-11-19 21:09:26  TIGER2024/ROADS/
              2026-11-18 12:54:41  TIGER2024/STATE/
        2504  2026-11-17 05:07:06  .gitattributes
      379192  2026-11-18 18:26:12  ACS2025_Table_Shells.parquet
          28  2026-11-17 05:07:06  README.md
      959136  2026-11-19 21:08:13  TIGER2024/ROADS/tl_2024_01001_roads.zip
     2325557  2026-11-19 21:08:13  TIGER2024/ROADS/tl_2024_01003_roads.zip
      682701  2026-11-19 21:08:13  TIGER2024/ROADS/tl_2024_01005_roads.zip
      760758  2026-11-19 21:08:13  TIGER2024/ROADS/tl_2024_01007_roads.zip
     1326049  2026-11-19 21:08:13  TIGER2024/ROADS/tl_2024_01009_roads.zip
      460118  2026-11-19 21:08:13  TIGER2024/ROADS/tl_2024_01011_roads.zip
      689675  2026-11-19 21:08:13  TIGER2024/ROADS/tl_2024_01013_roads.zip
     1538584  2026-11-19 21:08:13  TIGER2024/ROADS/tl_2024_01015_roads.zip
     1042564  2026-11-19 21:08:13  TIGER2024/ROADS/tl_2024_01017_roads.zip
     1217376  20